In [7]:
# ============================================================
# Visualization dependencies
# Works best in Google Colab, JupyterLab, or classic Jupyter.
# ============================================================

import sys
import subprocess
import importlib.util

def install_if_missing(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
    else:
        print(f"{package_name} already installed.")

install_if_missing("py3Dmol")
install_if_missing("ipywidgets")
install_if_missing("matplotlib")
install_if_missing("numpy")

print("Visualization setup complete.")

py3Dmol already installed.
ipywidgets already installed.
matplotlib already installed.
numpy already installed.
Visualization setup complete.


In [8]:
# ============================================================
# Interactive H2 molecule visualization with py3Dmol
# ============================================================

import py3Dmol
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

def h2_xyz(bond_length_angstrom: float) -> str:
    """
    Return an XYZ-format string for H2 with a chosen H-H bond length.

    The two hydrogen atoms are placed symmetrically around the origin
    along the z-axis.
    """
    z = bond_length_angstrom / 2
    xyz = f"""2
H2 molecule, H-H bond length = {bond_length_angstrom:.3f} Angstrom
H 0.000000 0.000000 {-z:.6f}
H 0.000000 0.000000 { z:.6f}
"""
    return xyz

def show_h2(bond_length_angstrom=0.741):
    """
    Render an interactive H2 molecule using py3Dmol.
    The equilibrium bond length of H2 is about 0.741 Angstrom.
    """
    view = py3Dmol.view(width=520, height=380)
    view.addModel(h2_xyz(bond_length_angstrom), "xyz")
    view.setStyle({
        "sphere": {"scale": 0.35},
        "stick": {"radius": 0.08}
    })
    view.setBackgroundColor("white")
    view.zoomTo()
    view.show()

bond_slider = widgets.FloatSlider(
    value=0.741,
    min=0.30,
    max=2.50,
    step=0.01,
    description="H-H distance / Å",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

output = widgets.Output()

def update_h2(change=None):
    with output:
        clear_output(wait=True)
        show_h2(bond_slider.value)
        print(
            f"H-H bond length: {bond_slider.value:.3f} Å\n"
            "Each geometry corresponds to a molecular Hamiltonian H(R). "
            "The goal is to estimate the lowest eigenvalue, the ground-state energy, for that Hamiltonian."
        )

bond_slider.observe(update_h2, names="value")

display(bond_slider, output)
update_h2()

FloatSlider(value=0.741, continuous_update=False, description='H-H distance / Å', layout=Layout(width='500px')…

Output()

As the H-H bond length changes, the molecular Hamiltonian changes. For each geometry, the ground-state energy is the lowest eigenvalue of that Hamiltonian.

VQE approximates this ground-state energy by preparing a parameterized quantum state and minimizing the measured expectation value of the Hamiltonian.

In [9]:
# ============================================================
# Toy protein-folding visualization:
# Coarse-grained lattice model with a simple energy function
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from IPython.display import display, clear_output

# -----------------------------
# 1. Define a toy amino-acid chain
# -----------------------------
# H = hydrophobic bead, P = polar bead.
# This is not a real protein sequence. It is a teaching model.
sequence = "HPPHHPHPPHPH"
n_beads = len(sequence)

# Initial self-avoiding chain on a 3D cubic lattice
initial_coords = np.array([[i, 0, 0] for i in range(n_beads)], dtype=int)

def is_self_avoiding(coords):
    """Check whether all lattice positions are unique."""
    return len({tuple(c) for c in coords}) == len(coords)

def lattice_neighbors(a, b):
    """Return True if two lattice points are nearest neighbors."""
    return np.sum(np.abs(a - b)) == 1

def folding_energy(coords, sequence):
    """
    Simple HP-like energy function.

    Contributions:
    - Non-consecutive H-H nearest-neighbor contacts: -1.0 each
    - Mild bending penalty: +0.05 for sharp turns
    - Mild compactness reward: proportional to radius of gyration

    This is pedagogical, not a chemically accurate force field.
    """
    energy = 0.0
    n = len(sequence)

    # Hydrophobic contact reward
    for i in range(n):
        for j in range(i + 2, n):  # exclude bonded neighbors
            if sequence[i] == "H" and sequence[j] == "H":
                if lattice_neighbors(coords[i], coords[j]):
                    energy -= 1.0

    # Mild bending penalty
    for i in range(1, n - 1):
        v1 = coords[i] - coords[i - 1]
        v2 = coords[i + 1] - coords[i]
        if not np.array_equal(v1, v2):
            energy += 0.05

    # Mild compactness term
    center = coords.mean(axis=0)
    radius_of_gyration = np.sqrt(np.mean(np.sum((coords - center) ** 2, axis=1)))
    energy += 0.03 * radius_of_gyration

    return float(energy)

def random_rotation_matrix_90():
    """
    Generate one of the 90-degree lattice rotations around x, y, or z.
    """
    axis = np.random.choice(["x", "y", "z"])
    angle = np.random.choice([1, 2, 3]) * np.pi / 2

    c = int(round(np.cos(angle)))
    s = int(round(np.sin(angle)))

    if axis == "x":
        return np.array([[1, 0, 0],
                         [0, c, -s],
                         [0, s,  c]], dtype=int)
    if axis == "y":
        return np.array([[ c, 0, s],
                         [ 0, 1, 0],
                         [-s, 0, c]], dtype=int)
    return np.array([[c, -s, 0],
                     [s,  c, 0],
                     [0,  0, 1]], dtype=int)

def pivot_move(coords):
    """
    Apply a random pivot move to part of the chain.
    This preserves bond lengths on the cubic lattice.
    """
    n = len(coords)
    pivot = np.random.randint(1, n - 1)
    R = random_rotation_matrix_90()

    new_coords = coords.copy()
    pivot_point = coords[pivot]

    # Rotate the tail after the pivot
    tail = coords[pivot + 1:] - pivot_point
    rotated_tail = tail @ R.T
    new_coords[pivot + 1:] = pivot_point + rotated_tail

    return new_coords

def simulated_annealing_folding(
    initial_coords,
    sequence,
    steps=400,
    start_temp=2.0,
    end_temp=0.05,
    seed=7
):
    """
    Generate a trajectory of conformations using simulated annealing.
    """
    np.random.seed(seed)

    coords = initial_coords.copy()
    energy = folding_energy(coords, sequence)

    trajectory = [coords.copy()]
    energies = [energy]
    best_coords = coords.copy()
    best_energy = energy

    for step in range(steps):
        temp = start_temp * (end_temp / start_temp) ** (step / max(steps - 1, 1))

        proposal = pivot_move(coords)

        if not is_self_avoiding(proposal):
            trajectory.append(coords.copy())
            energies.append(energy)
            continue

        proposal_energy = folding_energy(proposal, sequence)
        delta = proposal_energy - energy

        accept = delta < 0 or np.random.rand() < np.exp(-delta / temp)

        if accept:
            coords = proposal
            energy = proposal_energy

        if energy < best_energy:
            best_energy = energy
            best_coords = coords.copy()

        trajectory.append(coords.copy())
        energies.append(energy)

    return trajectory, np.array(energies), best_coords, best_energy

trajectory, energies, best_coords, best_energy = simulated_annealing_folding(
    initial_coords,
    sequence,
    steps=500,
    start_temp=2.5,
    end_temp=0.03,
    seed=11
)

print("Toy sequence:", sequence)
print("Initial energy:", energies[0])
print("Best energy found:", best_energy)
print("Number of conformations in trajectory:", len(trajectory))

Toy sequence: HPPHHPHPPHPH
Initial energy: 0.10356157588603988
Best energy found: -4.612085622779744
Number of conformations in trajectory: 501


In [10]:
# ============================================================
# Interactive folding trajectory viewer
# ============================================================

def plot_conformation(coords, sequence, energy, step, best_energy):
    """
    Plot a 3D coarse-grained conformation and energy trace.
    """
    fig = plt.figure(figsize=(12, 5))

    # 3D conformation plot
    ax = fig.add_subplot(1, 2, 1, projection="3d")

    xs, ys, zs = coords[:, 0], coords[:, 1], coords[:, 2]

    # Backbone
    ax.plot(xs, ys, zs, linewidth=2)

    # Beads
    for i, aa in enumerate(sequence):
        marker_size = 130 if aa == "H" else 90
        ax.scatter(xs[i], ys[i], zs[i], s=marker_size)
        ax.text(xs[i], ys[i], zs[i], f" {aa}{i}", fontsize=8)

    ax.set_title(f"Toy folded conformation\nstep = {step}, energy = {energy:.3f}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")

    # Equal-ish axis scaling
    max_range = np.array([
        xs.max() - xs.min(),
        ys.max() - ys.min(),
        zs.max() - zs.min()
    ]).max()
    if max_range == 0:
        max_range = 1

    mid_x = (xs.max() + xs.min()) / 2
    mid_y = (ys.max() + ys.min()) / 2
    mid_z = (zs.max() + zs.min()) / 2

    ax.set_xlim(mid_x - max_range / 2 - 1, mid_x + max_range / 2 + 1)
    ax.set_ylim(mid_y - max_range / 2 - 1, mid_y + max_range / 2 + 1)
    ax.set_zlim(mid_z - max_range / 2 - 1, mid_z + max_range / 2 + 1)

    # Energy trace
    ax2 = fig.add_subplot(1, 2, 2)
    ax2.plot(energies, linewidth=1.5)
    ax2.scatter([step], [energies[step]], s=60)
    ax2.axhline(best_energy, linestyle="--", label=f"best energy = {best_energy:.3f}")
    ax2.set_xlabel("Annealing step")
    ax2.set_ylabel("Toy energy")
    ax2.set_title("Energy decreases as lower-energy conformations are found")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

step_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(trajectory) - 1,
    step=1,
    description="Folding step",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px")
)

fold_output = widgets.Output()

def update_folding(change=None):
    step = step_slider.value
    with fold_output:
        clear_output(wait=True)
        plot_conformation(
            trajectory[step],
            sequence,
            energies[step],
            step,
            best_energy
        )
        print(
            "Interpretation: this is a simplified coarse-grained folding model. "
            "Each conformation has an energy. The search process tries to find lower-energy structures, "
            "analogous to how VQE searches for a low-energy quantum state of a Hamiltonian."
        )

step_slider.observe(update_folding, names="value")

display(step_slider, fold_output)
update_folding()

IntSlider(value=0, continuous_update=False, description='Folding step', layout=Layout(width='600px'), max=500,…

Output()

## Visual hook: protein folding as an energy-landscape problem

Protein folding is often described as a search over many possible conformations, where each conformation has an associated energy. The physically relevant folded structure is often related to a low-energy or ground-state configuration.

This visualization is not a realistic molecular-dynamics simulation. It is a coarse-grained lattice model intended to communicate the optimization idea: many configurations, an energy function, and a search process that tries to approach lower-energy conformations.

This connects naturally to VQE: in VQE, the object being minimized is not a classical folding score but the expectation value of a quantum Hamiltonian.

In [11]:
# ============================================================
# Render one toy folded conformation with py3Dmol
# ============================================================

import py3Dmol

def coarse_chain_xyz(coords, sequence):
    """
    Convert a coarse-grained bead chain into an XYZ-like string.

    Hydrophobic beads are represented as carbon atoms.
    Polar beads are represented as oxygen atoms.

    This is a visual metaphor, not a chemically meaningful molecule.
    """
    lines = [str(len(sequence)), "Toy coarse-grained protein conformation"]

    for aa, (x, y, z) in zip(sequence, coords):
        element = "C" if aa == "H" else "O"
        lines.append(f"{element} {float(x):.3f} {float(y):.3f} {float(z):.3f}")

    return "\n".join(lines)

def show_coarse_chain_py3dmol(step=None):
    if step is None:
        step = int(np.argmin(energies))

    coords = trajectory[step]
    xyz = coarse_chain_xyz(coords, sequence)

    view = py3Dmol.view(width=600, height=420)
    view.addModel(xyz, "xyz")
    view.setStyle({"sphere": {"scale": 0.35}, "stick": {"radius": 0.08}})
    view.setBackgroundColor("white")
    view.zoomTo()
    view.show()

    print(f"Toy conformation step: {step}")
    print(f"Toy energy: {energies[step]:.3f}")
    print("C = hydrophobic bead, O = polar bead in this visual metaphor.")

show_coarse_chain_py3dmol()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Toy conformation step: 400
Toy energy: -4.612
C = hydrophobic bead, O = polar bead in this visual metaphor.
